# Sentiment task

Comparing three sentiment systems on the project test set: vader (rule based), naive bayes
with scikit-learn, and a pretrained transformer. this is the task we go deeper on so there are
a couple of extra runs at the end (different naive bayes settings + the binary bert model).

### downloads
run once.

In [14]:
import sys
print(sys.executable)

c:\Users\NAD\anaconda3\envs\tm311\python.exe


In [15]:
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("downloads ready")

downloads ready


### settings

In [16]:
from pathlib import Path

# project test file
TEST_PATH = "Sentiment-topic-test.tsv"

# airline tweets training data, one folder per class
# from the repo: lab_sessions/lab3/airlinetweets.zip (next cell unzips it)
AIRLINE_ZIP = "airlinetweets.zip"
AIRLINE_DIR = "airlinetweets"

# 3 labels, alphabetical like sklearn
LABELS = ["negative", "neutral", "positive"]
print("settings loaded")

settings loaded


### load data
test set (10 sentences) and the airline tweets used to train naive bayes.

In [17]:
import pandas as pd

# test set
test_df = pd.read_csv(TEST_PATH, sep="\t")
# remove windows line endings
for col in test_df.columns:
    if test_df[col].dtype == object:
        test_df[col] = test_df[col].str.replace("\r", "", regex=False).str.strip()

TEST_TEXTS = test_df["text"].tolist()
GOLD_SENT  = test_df["sentiment"].tolist()

print("TEST SET")
print("number of sentences:", len(test_df))
print("\nsentiment distribution in test set:")
print(test_df["sentiment"].value_counts())
print("\ntopic distribution in test set (for your topic task later):")
print(test_df["topic"].value_counts())
test_df.head()

TEST SET
number of sentences: 10

sentiment distribution in test set:
sentiment
positive    4
negative    3
neutral     3
Name: count, dtype: int64

topic distribution in test set (for your topic task later):
topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64


,sentence id,text,sentiment,topic
0,0,It took eight years for Warner Brothers to rec...,negative,movie
1,1,All the New York University students love this...,positive,restaurant
2,2,This Italian place is really trendy but they h...,negative,restaurant
3,3,"In conclusion, my review of this book would be...",positive,book
4,4,The story of this movie is focused on Carl Bra...,neutral,movie


In [18]:
import zipfile, os

# unzip if not done already
if not os.path.isdir(AIRLINE_DIR) and os.path.isfile(AIRLINE_ZIP):
    with zipfile.ZipFile(AIRLINE_ZIP) as z:
        z.extractall(".")
    print("unzipped airline tweets")

from sklearn.datasets import load_files

# load_files reads one folder per class
airline = load_files(AIRLINE_DIR, categories=LABELS, encoding="utf-8", decode_error="ignore")

print("TRAINING SET (airline tweets)")
print("classes (target_names):", airline.target_names)
print("total training tweets:", len(airline.data))
import collections
counts = collections.Counter(airline.target_names[t] for t in airline.target)
print("class balance:", dict(counts))

TRAINING SET (airline tweets)
classes (target_names): ['negative', 'neutral', 'positive']
total training tweets: 4755
class balance: {'neutral': 1515, 'positive': 1490, 'negative': 1750}


### vader (rule based)
vader gives positive/negative/neutral scores plus a compound score. I use the compound:
>= 0.05 positive, <= -0.05 negative, else neutral.

In [19]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader_model = SentimentIntensityAnalyzer()

def vader_label(text):
    compound = vader_model.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

vader_preds = [vader_label(t) for t in TEST_TEXTS]
print("VADER predictions:")
for t, p in zip(TEST_TEXTS, vader_preds):
    print(f"  [{p:8}]  {t[:70]}")

VADER predictions:
  [negative]  It took eight years for Warner Brothers to recover from the disaster t
  [positive]  All the New York University students love this diner in Soho so it mak
  [positive]  This Italian place is really trendy but they have forgotten about the 
  [positive]  In conclusion, my review of this book would be: I like Jane Austen and
  [positive]  The story of this movie is focused on Carl Brashear played by Cuba Goo
  [positive]  Chris O'Donnell stated that while filming for this movie, he felt like
  [positive]  My husband and I moved to Amsterdam 6 years ago and for as long as we 
  [positive]  Dame Maggie Smith performed her role excellently, as she does in all h
  [neutral ]  The new movie by Mr. Kruno was shot in New York, but the story takes p
  [positive]  I always have loved English novels, but I just couldn't get into this 


### naive bayes (scikit-learn)
turn the tweets into word features (nltk tokenizer, english stopwords removed, min_df=2),
weight with tf-idf, then multinomial naive bayes trained on the airline tweets.
USE_TFIDF and MIN_DF are the settings I change later for the comparison.

In [20]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB

# settings I change for the comparison
USE_TFIDF = True   # tf-idf or plain counts
MIN_DF    = 2      # try 2, 5, 10

vectorizer = CountVectorizer(min_df=MIN_DF,
                             tokenizer=nltk.word_tokenize,
                             stop_words=stopwords.words("english"))

# fit on the training tweets, transform both
train_counts = vectorizer.fit_transform(airline.data)
test_counts  = vectorizer.transform(TEST_TEXTS)

if USE_TFIDF:
    tfidf = TfidfTransformer()
    train_X = tfidf.fit_transform(train_counts)
    test_X  = tfidf.transform(test_counts)
else:
    train_X, test_X = train_counts, test_counts

nb = MultinomialNB().fit(train_X, airline.target)

# numbers back to label names
nb_pred_ids = nb.predict(test_X)
nb_preds = [airline.target_names[i] for i in nb_pred_ids]

print(f"Naive Bayes settings: USE_TFIDF={USE_TFIDF}, MIN_DF={MIN_DF}")
for t, p in zip(TEST_TEXTS, nb_preds):
    print(f"  [{p:8}]  {t[:70]}")

c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


Naive Bayes settings: USE_TFIDF=True, MIN_DF=2
  [negative]  It took eight years for Warner Brothers to recover from the disaster t
  [positive]  All the New York University students love this diner in Soho so it mak
  [negative]  This Italian place is really trendy but they have forgotten about the 
  [negative]  In conclusion, my review of this book would be: I like Jane Austen and
  [neutral ]  The story of this movie is focused on Carl Brashear played by Cuba Goo
  [positive]  Chris O'Donnell stated that while filming for this movie, he felt like
  [negative]  My husband and I moved to Amsterdam 6 years ago and for as long as we 
  [negative]  Dame Maggie Smith performed her role excellently, as she does in all h
  [negative]  The new movie by Mr. Kruno was shot in New York, but the story takes p
  [negative]  I always have loved English novels, but I just couldn't get into this 


### top features per class
which words naive bayes links to each class, helps for the analysis.

In [21]:
def important_features_per_class(vectorizer, classifier, n=10):
    feature_names = vectorizer.get_feature_names_out()
    for class_index, class_label in enumerate(classifier.classes_):
        topn = sorted(zip(classifier.feature_count_[class_index], feature_names),
                      reverse=True)[:n]
        print(f"\nTop {n} words for class '{airline.target_names[class_label]}':")
        for count, feat in topn:
            print(f"   {feat}")

# clearer with bag-of-words but works either way
important_features_per_class(vectorizer, nb, n=10)


Top 10 words for class 'negative':
   united
   .
   @
   ``
   flight
   ?
   #
   !
   n't
   ''

Top 10 words for class 'neutral':
   @
   ?
   jetblue
   southwestair
   .
   ``
   americanair
   :
   usairways
   flight

Top 10 words for class 'positive':
   !
   @
   .
   thanks
   thank
   jetblue
   southwestair
   ``
   #
   americanair


### transformer
load a pretrained model with the huggingface pipeline. the lab model (distilbert sst-2) only
does positive/negative with no neutral, and our test set has 3 classes, so I use a 3-class
twitter model instead. the binary one is run further down to compare.

In [22]:
from transformers import pipeline

# 3-class model, gives neg/neu/pos
bert = pipeline("sentiment-analysis",
                model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# model can return label names or LABEL_0/1/2
label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive",
             "negative": "negative", "neutral": "neutral", "positive": "positive"}

bert_raw = bert(TEST_TEXTS)
bert_preds = [label_map[r["label"]] for r in bert_raw]

print("Transformer (twitter-roberta, 3-class) predictions:")
for t, p in zip(TEST_TEXTS, bert_preds):
    print(f"  [{p:8}]  {t[:70]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16752.21it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer (twitter-roberta, 3-class) predictions:
  [negative]  It took eight years for Warner Brothers to recover from the disaster t
  [positive]  All the New York University students love this diner in Soho so it mak
  [negative]  This Italian place is really trendy but they have forgotten about the 
  [positive]  In conclusion, my review of this book would be: I like Jane Austen and
  [neutral ]  The story of this movie is focused on Carl Brashear played by Cuba Goo
  [neutral ]  Chris O'Donnell stated that while filming for this movie, he felt like
  [positive]  My husband and I moved to Amsterdam 6 years ago and for as long as we 
  [positive]  Dame Maggie Smith performed her role excellently, as she does in all h
  [neutral ]  The new movie by Mr. Kruno was shot in New York, but the story takes p
  [negative]  I always have loved English novels, but I just couldn't get into this 


In [23]:
# binary distilbert for contrast. no neutral class so I use a confidence band for neutral.

bert2 = pipeline("sentiment-analysis",
                model="distilbert-base-uncased-finetuned-sst-2-english")
def distilbert_label(text, neutral_band=0.65):
    r = bert2(text)[0]
    if r["score"] < neutral_band:      # not confident -> call it neutral
        return "neutral"
    return r["label"].lower()          # 'positive' or 'negative'
bert2_preds = [distilbert_label(t) for t in TEST_TEXTS]
print(bert2_preds)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3635.00it/s]


['negative', 'positive', 'negative', 'positive', 'positive', 'negative', 'positive', 'positive', 'negative', 'negative']


### evaluation
classification report (precision/recall/f1) + confusion matrix per system.

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(name, preds):
    print("=" * 60)
    print("SYSTEM:", name)
    print("=" * 60)
    print(classification_report(GOLD_SENT, preds, labels=LABELS, zero_division=0))
    print("Confusion matrix (rows = gold, cols = predicted), order:", LABELS)
    print(confusion_matrix(GOLD_SENT, preds, labels=LABELS))
    print()

evaluate("VADER (rule-based)", vader_preds)
evaluate("Naive Bayes (scikit-learn)", nb_preds)
evaluate("Transformer (twitter-roberta)", bert_preds)

SYSTEM: VADER (rule-based)
              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.83      0.56      0.56        10
weighted avg       0.80      0.60      0.57        10

Confusion matrix (rows = gold, cols = predicted), order: ['negative', 'neutral', 'positive']
[[1 0 2]
 [0 1 2]
 [0 0 4]]

SYSTEM: Naive Bayes (scikit-learn)
              precision    recall  f1-score   support

    negative       0.43      1.00      0.60         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      0.25      0.33         4

    accuracy                           0.50        10
   macro avg       0.64      0.53      0.48        10
weighted avg       0.63      0.50      0.46        10

Confusion matrix (rows = gold, cols = predicted), orde

### per sentence comparison
only 10 sentences so I can check every prediction. saved to csv for the error analysis.

In [25]:
compare = pd.DataFrame({
    "id": test_df["sentence id"],
    "text": TEST_TEXTS,
    "gold": GOLD_SENT,
    "vader": vader_preds,
    "naive_bayes": nb_preds,
    "transformer": bert_preds,
})
compare["vader_ok"]       = compare["vader"]       == compare["gold"]
compare["naive_bayes_ok"] = compare["naive_bayes"] == compare["gold"]
compare["transformer_ok"] = compare["transformer"] == compare["gold"]

compare.to_csv("sentiment_predictions_comparison.csv", index=False)
print("saved -> sentiment_predictions_comparison.csv")
compare

saved -> sentiment_predictions_comparison.csv


,id,text,gold,vader,naive_bayes,transformer,vader_ok,naive_bayes_ok,transformer_ok
0,0,It took eight years for Warner Brothers to rec...,negative,negative,negative,negative,True,True,True
1,1,All the New York University students love this...,positive,positive,positive,positive,True,True,True
2,2,This Italian place is really trendy but they h...,negative,positive,negative,negative,False,True,True
3,3,"In conclusion, my review of this book would be...",positive,positive,negative,positive,True,False,True
4,4,The story of this movie is focused on Carl Bra...,neutral,positive,neutral,neutral,False,True,True
5,5,Chris O'Donnell stated that while filming for ...,neutral,positive,positive,neutral,False,False,True
6,6,My husband and I moved to Amsterdam 6 years ag...,positive,positive,negative,positive,True,False,True
7,7,Dame Maggie Smith performed her role excellent...,positive,positive,negative,positive,True,False,True
8,8,The new movie by Mr. Kruno was shot in New Yor...,neutral,neutral,negative,neutral,True,False,True
9,9,"I always have loved English novels, but I just...",negative,positive,negative,negative,False,True,True


### naive bayes settings comparison
re-run naive bayes with tf-idf vs bag-of-words and min_df 2/5/10, print macro-f1 for each.

In [26]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
import nltk
from nltk.corpus import stopwords

def run_nb(use_tfidf, min_df):
    vec = CountVectorizer(min_df=min_df, tokenizer=nltk.word_tokenize,
                          stop_words=stopwords.words("english"))
    Xtr = vec.fit_transform(airline.data)
    Xte = vec.transform(TEST_TEXTS)
    if use_tfidf:
        tf = TfidfTransformer(); Xtr = tf.fit_transform(Xtr); Xte = tf.transform(Xte)
    clf = MultinomialNB().fit(Xtr, airline.target)
    preds = [airline.target_names[i] for i in clf.predict(Xte)]
    return f1_score(GOLD_SENT, preds, labels=LABELS, average="macro")

for ut in [True, False]:
    for md in [2, 5, 10]:
        print(f"TF-IDF={ut}, min_df={md}  ->  macro-F1 = {run_nb(ut, md):.2f}")

c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


TF-IDF=True, min_df=2  ->  macro-F1 = 0.48


c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


TF-IDF=True, min_df=5  ->  macro-F1 = 0.53


c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


...

TF-IDF=False, min_df=2  ->  macro-F1 = 0.47


c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


TF-IDF=False, min_df=5  ->  macro-F1 = 0.58


c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\NAD\anaconda3\envs\tm311\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


TF-IDF=False, min_df=10  ->  macro-F1 = 0.58
